# Controles de sigma cero y reconstrucción VAE

Diagnóstico sobre `potw1803a.tif`, con la misma referencia preparada y entrada
gaussiana conservadas. **No hay resultados reales de este control al preparar
el notebook.** Los ensayos previos de SD y Tile empeoraron las métricas respecto
de la entrada ruidosa; estos controles ayudan a estudiar dónde aparecen cambios.

Seleccioná **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**, si está
disponible, e interrumpí otros servidores Gradio antes de ejecutar. El notebook
clona GitHub y exige CUDA. Las pruebas locales no descargaron pesos ni ejecutaron
inferencias reales.

**Sigma cero** significa que no se agrega ruido artificial. La fotografía
publicada conserva su procesamiento y puede contener ruido propio. La intensidad
de difusión se mantiene en 0,10; sigma cero no significa `strength=0`.

## Qué se compara

1. Identidad de la referencia y entrada gaussiana, sin IA, como puntos de partida.
2. Referencia → VAE con moda del posterior: control determinista, sin muestreo.
3. Referencia → VAE con muestreo posterior y semilla 42, como el usado por img2img.
4. Entrada ruidosa → VAE con muestreo posterior y semilla 42.
5. Referencia sin ruido añadido → SD 1.5, intensidad 0,10, 30 pasos, semilla 42.
6. Misma entrada sigma cero → SD 1.5 + Tile, fuerza 1,0, igual intensidad y semilla.

En los controles VAE no se ejecutan UNet, scheduler ni ControlNet, aunque se
carga el pipeline para reutilizar su VAE, preprocesamiento y filtro de seguridad.
En el control Tile sigma cero, la referencia preparada es explícitamente la
propia entrada y condición: se estudia su alteración, no la restauración de
una copia ruidosa con una guía limpia externa.

**Límite:** los procesos son no lineales. No se pueden restar errores o asignar
porcentajes causales exactos de deterioro al VAE y la difusión. La moda VAE no
es el camino normal de muestreo de img2img. El filtro de seguridad sigue activo.

Fuente: https://esahubble.org/images/potw1803a/
Crédito: **NASA, ESA, and J. Comerford (University of Colorado-Boulder)**.
CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/ — según
https://esahubble.org/copyright/. Modificaciones: RGB, reducción, ruido en la
copia y, al ejecutar, VAE o difusión. No se verifica autenticidad científica.


In [ ]:
import os
import signal
import subprocess

def ejecutar_con_salida(comando, cwd=None):
    """Muestra stdout y stderr del proceso en la salida de la celda de VS Code."""
    entorno = os.environ.copy()
    entorno['PYTHONUNBUFFERED'] = '1'
    print('Proceso iniciado. Los mensajes aparecerán debajo de esta celda.', flush=True)
    with subprocess.Popen(
        comando, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1, env=entorno,
        start_new_session=(os.name == 'posix'),
    ) as proceso:
        try:
            for linea in proceso.stdout:
                print(linea, end='', flush=True)
            codigo = proceso.wait()
        except KeyboardInterrupt:
            if proceso.poll() is None:
                if os.name == 'posix':
                    os.killpg(proceso.pid, signal.SIGTERM)
                else:
                    proceso.terminate()
                try:
                    proceso.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    if os.name == 'posix':
                        os.killpg(proceso.pid, signal.SIGKILL)
                    else:
                        proceso.kill()
                    proceso.wait()
            raise
    if codigo != 0:
        raise subprocess.CalledProcessError(codigo, comando)
    print('Proceso finalizado correctamente.', flush=True)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys

if not Path('/content').is_dir():
    raise RuntimeError('Este notebook se ejecuta en Google Colab.')
proyecto = Path('/content') / ('astrorestore_controles_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch',
                'https://github.com/Sabri-Nang/astro-restauracion.git', str(proyecto)], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=proyecto, text=True).strip()
necesarios = ['src/diffusion.py', 'src/diagnostics.py', 'src/evaluation.py', 'src/experiment.py',
              'src/honesty.py', 'pyproject.toml', 'uv.lock',
              'assets/ruido_gaussiano/potw1803a_sigma0.03_seed123_size256/referencia_preparada.png',
              'assets/ruido_gaussiano/potw1803a_sigma0.03_seed123_size256/entrada_ruidosa.png']
for nombre in necesarios:
    if not (proyecto / nombre).is_file():
        raise ValueError('Falta publicar el archivo necesario: ' + nombre)
origen_codigo = {'revision_git': revision, 'sha256_archivos': {
    nombre: hashlib.sha256((proyecto / nombre).read_bytes()).hexdigest() for nombre in necesarios}}
print('Revisión Git:', revision)
# Reutilizar, si existe, una caché del ensayo Tile en esta misma sesión.
candidatas = sorted(p for p in Path('/content').glob('astrorestore_tile_*/.cache/huggingface') if p.is_dir())
cache = candidatas[-1] if candidatas else Path('/content/astrorestore_model_cache')
cache.mkdir(parents=True, exist_ok=True)
(proyecto / '.cache').mkdir(exist_ok=True)
(proyecto / '.cache/huggingface').symlink_to(cache, target_is_directory=True)
print('Caché de pesos:', cache)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.12.3'], check=True)
ejecutar_con_salida(['uv', 'sync', '--locked', '--python', '3.12'], cwd=proyecto)
ejecutar_con_salida(['uv', 'run', '--locked', 'python', '-c',
    "import sys,torch; assert sys.version_info[:2]==(3,12); assert torch.cuda.is_available(), 'Activá GPU en Colab'; print('GPU:',torch.cuda.get_device_name(0))"], cwd=proyecto)


In [ ]:
from IPython.display import display, Image, Markdown

activo = proyecto / 'assets/ruido_gaussiano/potw1803a_sigma0.03_seed123_size256'
display(Markdown('### Referencia: sigma artificial cero, no necesariamente libre de ruido real'))
display(Image(filename=str(activo / 'referencia_preparada.png')))
display(Markdown('### Copia con sigma 0,03, semilla de ruido 123'))
display(Image(filename=str(activo / 'entrada_ruidosa.png')))


In [ ]:
salida = proyecto / 'outputs/etapa3/controles_sigma0_vae'
ejecutar_con_salida(['uv', 'run', '--locked', 'python', '-u', '-m', 'src.diagnostics',
    '--asset-dir', str(activo), '--output-dir', str(salida), '--device', 'cuda'], cwd=proyecto)
registro = json.loads((salida / 'controles.json').read_text(encoding='utf-8'))
registro['origen_codigo'] = origen_codigo
(salida / 'controles.json').write_text(json.dumps(registro, ensure_ascii=False, indent=2) + chr(10), encoding='utf-8')
print('Fase:', registro['fase'])
for caso in registro['resultados']:
    print(caso['control'], caso['estado'], caso.get('frente_referencia', caso.get('error')))


In [ ]:
display(Image(filename=str(salida / 'regiones_controles.png')))
for caso in registro['resultados']:
    if caso['estado'] == 'correcto':
        display(Markdown('### ' + caso['control']))
        display(Image(filename=str(salida / caso['id'] / 'salida.png')))
print('PSNR mayor y SSIM mayor indican más similitud a esta referencia sintética.')
print('Identidad: PSNR infinito se registra como psnr_db=null y psnr_infinito=true.')
print('Comparar también frente_su_entrada, las regiones y los parámetros. No atribuir porcentajes de error a cada etapa.')


In [ ]:
import shutil
from google.colab import files

archivo = shutil.make_archive(str(salida.parent / 'controles_sigma0_vae'), 'zip', salida)
files.download(archivo)
print('Conservá controles_sigma0_vae.zip antes de cerrar Colab y compartilo para revisar los resultados.')
